# GE-YOLOv8 Benchmark on RSNA 2024 Axial T2 Dataset

This notebook trains the custom **GE-YOLOv8** model (from the paper "Deep learning-based automatic detection and grading of disk herniation") on the **RSNA 2024 Lumbar Spine Degenerative Classification** dataset.

**Key Features:**
- **Custom Architecture:** Uses CSP (Gradient Search) and ECAAttention (Efficient Channel Attention) modules
- **Data Pipeline:** Identical to YOLOv11 benchmark for fair comparison
- **3 Class Detection:** Spinal Canal Stenosis (Center), Left (Neural Foraminal & Subarticular), Right (Neural Foraminal & Subarticular)

**References:**
- [Yolov8-GS-ECA GitHub Repository](https://github.com/hxxbb/Yolov8-GS-ECA)
- [RSNA 2024 Lumbar Spine Competition](https://www.kaggle.com/competitions/rsna-2024-lumbar-spine-degenerative-classification)

## 1. Environment Setup

The GE-YOLOv8 model uses custom modules (`CSP` for Gradient Search, `ECAAttention` for Attention) that are NOT in the standard pip package. We need to clone the official repository and install it in editable mode.

In [ ]:
# Clone the GE-YOLOv8 repository
!git clone https://github.com/hxxbb/Yolov8-GS-ECA.git

# Install in editable mode to use custom modules
!cd Yolov8-GS-ECA && pip install -e .

In [ ]:
# Install additional dependencies
!pip install pydicom pandas numpy opencv-python-headless pyyaml tqdm

In [ ]:
# Verification: Ensure ultralytics imports from the local directory
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"Ultralytics location: {ultralytics.__file__}")

# Verify custom modules are available
from ultralytics.nn.modules import ECAAttention, CSP
print("\n✅ Custom modules (ECAAttention, CSP) imported successfully!")

In [ ]:
# Import all required libraries
import os
import glob
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import pydicom
from tqdm import tqdm
import yaml

from ultralytics import YOLO

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

## 2. Data Pipeline (RSNA 2024 Axial T2)

This data pipeline is **IDENTICAL** to the YOLOv11 experiment for fair comparison.

**Key Processing Steps:**
- **Filter:** Only `series_description == 'Axial T2'`
- **Resize:** 384 x 384
- **Class Mapping (3 Classes):**
  - Class 0: Spinal Canal Stenosis (Center)
  - Class 1: Left Neural Foraminal & Left Subarticular (Left)
  - Class 2: Right Neural Foraminal & Right Subarticular (Right)

### 2.1 Configuration

In [ ]:
# Configuration
class Config:
    # Paths - Kaggle specific
    COMPETITION_DATA = '/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification'
    OUTPUT_DIR = '/kaggle/working/datasets/axial_t2'
    
    # Data processing
    TARGET_SIZE = (384, 384)  # Same as YOLOv11 benchmark
    BBOX_SIZE = 32  # Fixed bounding box size in pixels
    SERIES_FILTER = 'Axial T2'
    
    # Train/Val split
    VAL_RATIO = 0.2
    
    # Class mapping: 3 classes
    # Class 0: Spinal Canal Stenosis (Center)
    # Class 1: Left Neural Foraminal & Left Subarticular (Left)
    # Class 2: Right Neural Foraminal & Right Subarticular (Right)
    CLASS_MAPPING = {
        'spinal_canal_stenosis': 0,
        'left_neural_foraminal_narrowing': 1,
        'left_subarticular_stenosis': 1,
        'right_neural_foraminal_narrowing': 2,
        'right_subarticular_stenosis': 2,
    }
    
    CLASS_NAMES = ['center', 'left', 'right']
    NUM_CLASSES = 3

cfg = Config()

### 2.2 Load Metadata

In [ ]:
# Load CSV files
train_df = pd.read_csv(f'{cfg.COMPETITION_DATA}/train.csv')
train_label_coords = pd.read_csv(f'{cfg.COMPETITION_DATA}/train_label_coordinates.csv')
train_series_desc = pd.read_csv(f'{cfg.COMPETITION_DATA}/train_series_descriptions.csv')

print(f"Train samples: {len(train_df)}")
print(f"Label coordinates: {len(train_label_coords)}")
print(f"Series descriptions: {len(train_series_desc)}")

In [ ]:
# Filter for Axial T2 series only
axial_t2_series = train_series_desc[train_series_desc['series_description'] == cfg.SERIES_FILTER]
print(f"\nAxial T2 series count: {len(axial_t2_series)}")

# Merge to get label coordinates for Axial T2 series
axial_t2_labels = train_label_coords.merge(
    axial_t2_series[['study_id', 'series_id']], 
    on=['study_id', 'series_id'], 
    how='inner'
)
print(f"Axial T2 labels: {len(axial_t2_labels)}")

### 2.3 DICOM Processing Functions

In [ ]:
def load_dicom(dicom_path):
    """
    Load DICOM file and normalize to 8-bit.
    
    Args:
        dicom_path: Path to DICOM file
        
    Returns:
        Normalized 8-bit numpy array
    """
    dcm = pydicom.dcmread(dicom_path)
    img = dcm.pixel_array.astype(np.float32)
    
    # Normalize to 0-255
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
    img = img.astype(np.uint8)
    
    return img


def resize_image(img, target_size):
    """
    Resize image to target size.
    
    Args:
        img: Input image
        target_size: Tuple (width, height)
        
    Returns:
        Resized image and scale factors
    """
    h, w = img.shape[:2]
    resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    scale_x = target_size[0] / w
    scale_y = target_size[1] / h
    return resized, scale_x, scale_y


def point_to_bbox(x, y, img_width, img_height, bbox_size):
    """
    Convert point coordinates to YOLO format bounding box.
    
    Args:
        x, y: Point coordinates
        img_width, img_height: Image dimensions
        bbox_size: Fixed bounding box size in pixels
        
    Returns:
        YOLO format bbox (x_center, y_center, width, height) - normalized
    """
    # Center coordinates (normalized)
    x_center = x / img_width
    y_center = y / img_height
    
    # Box dimensions (normalized)
    box_w = bbox_size / img_width
    box_h = bbox_size / img_height
    
    # Clamp to valid range
    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    box_w = min(box_w, min(x_center, 1 - x_center) * 2)
    box_h = min(box_h, min(y_center, 1 - y_center) * 2)
    
    return x_center, y_center, box_w, box_h


def get_condition_class(condition):
    """
    Map condition name to class ID.
    
    Class mapping:
    - 0: Spinal Canal Stenosis (Center)
    - 1: Left Neural Foraminal & Left Subarticular (Left)
    - 2: Right Neural Foraminal & Right Subarticular (Right)
    """
    condition_lower = condition.lower().replace(' ', '_')
    return cfg.CLASS_MAPPING.get(condition_lower, None)

### 2.4 Process Dataset

In [ ]:
def process_dataset(axial_t2_labels, cfg):
    """
    Process the RSNA dataset and create YOLO format dataset.
    
    Label propagation: Labels are shared to slice n-1 and n+1.
    """
    # Create output directories
    for split in ['train', 'val']:
        os.makedirs(f'{cfg.OUTPUT_DIR}/{split}/images', exist_ok=True)
        os.makedirs(f'{cfg.OUTPUT_DIR}/{split}/labels', exist_ok=True)
    
    # Group labels by study and series
    grouped = axial_t2_labels.groupby(['study_id', 'series_id'])
    
    # Get unique study IDs for train/val split
    study_ids = axial_t2_labels['study_id'].unique()
    random.shuffle(study_ids.tolist())
    
    val_size = int(len(study_ids) * cfg.VAL_RATIO)
    val_study_ids = set(study_ids[:val_size])
    train_study_ids = set(study_ids[val_size:])
    
    print(f"Train studies: {len(train_study_ids)}, Val studies: {len(val_study_ids)}")
    
    stats = {'train': {'images': 0, 'labels': 0}, 'val': {'images': 0, 'labels': 0}}
    
    for (study_id, series_id), group in tqdm(grouped, desc="Processing series"):
        # Determine split
        split = 'val' if study_id in val_study_ids else 'train'
        
        # Get series path
        series_path = f"{cfg.COMPETITION_DATA}/train_images/{study_id}/{series_id}"
        
        if not os.path.exists(series_path):
            continue
        
        # Get all DICOM files in series
        dicom_files = sorted(glob.glob(f"{series_path}/*.dcm"))
        if not dicom_files:
            continue
        
        # Create instance number to file mapping
        instance_to_file = {}
        for dcm_path in dicom_files:
            try:
                dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
                instance_num = int(dcm.InstanceNumber)
                instance_to_file[instance_num] = dcm_path
            except Exception:
                continue
        
        # Group labels by instance number
        instance_labels = {}
        for _, row in group.iterrows():
            instance_num = int(row['instance_number'])
            
            # Propagate labels to n-1, n, n+1
            for offset in [-1, 0, 1]:
                target_instance = instance_num + offset
                if target_instance not in instance_labels:
                    instance_labels[target_instance] = []
                
                class_id = get_condition_class(row['condition'])
                if class_id is not None:
                    instance_labels[target_instance].append({
                        'class_id': class_id,
                        'x': row['x'],
                        'y': row['y']
                    })
        
        # Process each instance with labels
        for instance_num, labels in instance_labels.items():
            if instance_num not in instance_to_file:
                continue
            
            dcm_path = instance_to_file[instance_num]
            
            try:
                # Load and process image
                img = load_dicom(dcm_path)
                orig_h, orig_w = img.shape[:2]
                
                # Resize
                img_resized, scale_x, scale_y = resize_image(img, cfg.TARGET_SIZE)
                
                # Convert grayscale to RGB
                if len(img_resized.shape) == 2:
                    img_resized = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
                
                # Generate unique filename
                filename = f"{study_id}_{series_id}_{instance_num}"
                
                # Save image
                img_path = f"{cfg.OUTPUT_DIR}/{split}/images/{filename}.jpg"
                cv2.imwrite(img_path, img_resized)
                stats[split]['images'] += 1
                
                # Create YOLO format labels
                label_lines = []
                for label in labels:
                    # Scale coordinates
                    x_scaled = label['x'] * scale_x
                    y_scaled = label['y'] * scale_y
                    
                    # Convert to YOLO bbox format
                    x_center, y_center, box_w, box_h = point_to_bbox(
                        x_scaled, y_scaled,
                        cfg.TARGET_SIZE[0], cfg.TARGET_SIZE[1],
                        cfg.BBOX_SIZE
                    )
                    
                    label_lines.append(f"{label['class_id']} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
                    stats[split]['labels'] += 1
                
                # Save labels
                label_path = f"{cfg.OUTPUT_DIR}/{split}/labels/{filename}.txt"
                with open(label_path, 'w') as f:
                    f.write('\n'.join(label_lines))
                    
            except Exception as e:
                print(f"Error processing {dcm_path}: {e}")
                continue
    
    return stats

In [ ]:
# Process the dataset
print("Processing RSNA 2024 Axial T2 dataset...")
stats = process_dataset(axial_t2_labels, cfg)

print("\n" + "="*50)
print("Dataset Processing Complete!")
print("="*50)
print(f"\nTrain set: {stats['train']['images']} images, {stats['train']['labels']} labels")
print(f"Val set: {stats['val']['images']} images, {stats['val']['labels']} labels")

## 3. Model Configuration

Create the `data.yaml` file and load the custom GE-YOLOv8 architecture.

### 3.1 Create data.yaml

In [ ]:
# Create data.yaml for YOLO training
data_yaml = {
    'path': cfg.OUTPUT_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'nc': cfg.NUM_CLASSES,
    'names': cfg.CLASS_NAMES
}

data_yaml_path = f'{cfg.OUTPUT_DIR}/data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Created data.yaml at: {data_yaml_path}")
print("\nContents:")
with open(data_yaml_path, 'r') as f:
    print(f.read())

### 3.2 Load Custom GE-YOLOv8 Architecture

The custom model uses:
- **CSP** (Cross Stage Partial) blocks for Gradient Search
- **ECAAttention** (Efficient Channel Attention) for attention mechanism

In [ ]:
# Path to custom model YAML
MODEL_YAML = 'Yolov8-GS-ECA/models/v8/yolov8-ECA-CSP.yaml'

# Verify the model config exists
if os.path.exists(MODEL_YAML):
    print(f"✅ Found custom model config: {MODEL_YAML}")
    with open(MODEL_YAML, 'r') as f:
        print("\nModel Architecture:")
        print(f.read())
else:
    print(f"❌ Model config not found: {MODEL_YAML}")
    print("Make sure to clone the Yolov8-GS-ECA repository first!")

In [ ]:
# Modify the model YAML to use nc: 3 (our number of classes)
# We create a temporary modified config
import yaml

with open(MODEL_YAML, 'r') as f:
    model_config = yaml.safe_load(f)

# Update number of classes
model_config['nc'] = cfg.NUM_CLASSES

# Save modified config
modified_model_yaml = f'{cfg.OUTPUT_DIR}/yolov8-ECA-CSP-3class.yaml'
with open(modified_model_yaml, 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)

print(f"Created modified model config: {modified_model_yaml}")
print(f"Number of classes: {model_config['nc']}")

In [ ]:
# Initialize model from custom YAML (builds model from scratch with custom layers)
# Note: We initialize from YAML, not from yolov8n.pt, because the architecture is different
print("Initializing GE-YOLOv8 model from custom architecture...")
model = YOLO(modified_model_yaml)

# Print model information
print("\nModel Information:")
model.info()

## 4. Training (Benchmark Settings)

Training with the same settings as the YOLOv11 experiment for fair comparison:
- `imgsz=384`
- `epochs=50`
- `batch=16`
- `optimizer='AdamW'`
- `lr0=0.001`

In [ ]:
# Training configuration - same as YOLOv11 benchmark
TRAIN_CONFIG = {
    'data': data_yaml_path,
    'imgsz': 384,
    'epochs': 50,
    'batch': 16,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'project': '/kaggle/working/ge_yolov8_benchmark',
    'name': 'axial_t2_run',
    'exist_ok': True,
    'pretrained': False,  # Train from scratch with custom architecture
    'verbose': True,
    'device': 0,  # Use GPU
}

print("Training Configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")

In [ ]:
# Start training
print("\n" + "="*60)
print("Starting GE-YOLOv8 Training on RSNA 2024 Axial T2 Dataset")
print("="*60 + "\n")

results = model.train(**TRAIN_CONFIG)

## 5. Comparison Output

Display validation metrics for comparison with YOLOv11 results.

In [ ]:
# Load the best model for validation
best_model_path = f"{TRAIN_CONFIG['project']}/{TRAIN_CONFIG['name']}/weights/best.pt"
best_model = YOLO(best_model_path)

# Run validation
print("\nRunning final validation on best model...")
val_results = best_model.val(data=data_yaml_path, imgsz=384, batch=16)

In [ ]:
# Print comparison metrics
print("\n" + "="*70)
print("GE-YOLOv8 BENCHMARK RESULTS - RSNA 2024 Axial T2")
print("="*70)
print(f"\nModel Architecture: yolov8-ECA-CSP (CSP + ECAAttention)")
print(f"Dataset: RSNA 2024 Lumbar Spine - Axial T2 Only")
print(f"Image Size: {TRAIN_CONFIG['imgsz']}")
print(f"Epochs: {TRAIN_CONFIG['epochs']}")
print(f"Batch Size: {TRAIN_CONFIG['batch']}")
print(f"Optimizer: {TRAIN_CONFIG['optimizer']}")
print(f"Learning Rate: {TRAIN_CONFIG['lr0']}")

print("\n" + "-"*70)
print("VALIDATION METRICS (for comparison with YOLOv11):")
print("-"*70)

# Extract key metrics
metrics = val_results.results_dict

print(f"\n{'Metric':<30} {'Value':<15}")
print("-" * 45)
print(f"{'mAP50':<30} {metrics.get('metrics/mAP50(B)', 'N/A'):<15.4f}")
print(f"{'mAP50-95':<30} {metrics.get('metrics/mAP50-95(B)', 'N/A'):<15.4f}")
print(f"{'Precision':<30} {metrics.get('metrics/precision(B)', 'N/A'):<15.4f}")
print(f"{'Recall':<30} {metrics.get('metrics/recall(B)', 'N/A'):<15.4f}")

print("\n" + "-"*70)
print("PER-CLASS METRICS:")
print("-"*70)

# Per-class AP if available
for i, class_name in enumerate(cfg.CLASS_NAMES):
    ap50_key = f'metrics/mAP50(B)'
    print(f"  Class {i} ({class_name}): See detailed results above")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"\nBest model saved at: {best_model_path}")
print(f"Results directory: {TRAIN_CONFIG['project']}/{TRAIN_CONFIG['name']}")

In [ ]:
# Display training curves
from IPython.display import Image, display
import os

results_dir = f"{TRAIN_CONFIG['project']}/{TRAIN_CONFIG['name']}"

# Display results.png if it exists
results_img = f"{results_dir}/results.png"
if os.path.exists(results_img):
    print("Training Curves:")
    display(Image(filename=results_img))

# Display confusion matrix if it exists
confusion_matrix_img = f"{results_dir}/confusion_matrix.png"
if os.path.exists(confusion_matrix_img):
    print("\nConfusion Matrix:")
    display(Image(filename=confusion_matrix_img))

## Summary

This notebook trained the **GE-YOLOv8** model (with CSP and ECAAttention modules) on the RSNA 2024 Axial T2 dataset.

**Key Results:**
- The validation metrics (mAP50, mAP50-95) can now be directly compared with YOLOv11 results
- The data pipeline and training settings were kept identical for fair comparison

**Files Generated:**
- Dataset: `/kaggle/working/datasets/axial_t2/`
- Model weights: `/kaggle/working/ge_yolov8_benchmark/axial_t2_run/weights/best.pt`
- Training logs: `/kaggle/working/ge_yolov8_benchmark/axial_t2_run/`